In [ ]:
!pip install pylangacq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 2.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import pylangacq as pla
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# aphasiabank

In [ ]:
path_data = '/content/drive/MyDrive/tesis_monica/afasia/data/'
file_path = path_data + 'df_ES_clean_2.csv'
df_aphbank = pd.read_csv(file_path)
df_aphbank.head()

,mark_start,mark_end,transcriptions,sex,age,file,WAB_AQ,aphasia_type,WAB_AQ_category,fluency_speech,file_cut,duration,num_words
0,20585,24500,okay dímelo otra vez,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_20.585_3.915.wav,3.915,4
1,25710,26400,sí,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_25.71_0.69.wav,0.690,1
2,29400,31200,que piensa mi habla,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_29.4_1.8.wav,1.800,4
3,33248,40362,no es que flr flr decir las cosas a vveces ten...,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_33.248_7.114.wav,7.114,14
4,40362,47984,ahorita puedo hablar pero flr dos horas flr fl...,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_40.362_7.622.wav,7.622,14


In [ ]:
def extract_patient_info(file_directory):
    ds = pla.read_chat(file_directory)
    files = ds.file_paths()

    # Listas para almacenar la información extraída
    v_sex = []
    v_age = []
    v_WAB_AQ = []
    v_aphasia_type = []
    v_file_name = []

    for f in files:
        ds2 = pla.read_chat(f)
        header = ds2.headers()

        # Extraer información de los participantes
        sex = header[0]['Participants']['PAR']['sex']
        age = header[0]['Participants']['PAR']['age']
        WAB_AQ = header[0]['Participants']['PAR'].get('custom', None)
        aphasia_type = header[0]['Participants']['PAR']['group']

        # Extraer nombre del archivo
        file_name = f.split('/')[-1].replace('.cha', '.wav')

        # Añadir la información a las listas
        v_sex.append(sex)
        v_age.append(age[:2] if age else None)  # Quitar los meses
        v_WAB_AQ.append(WAB_AQ)
        v_aphasia_type.append(aphasia_type)
        v_file_name.append(file_name)

    # Crear un DataFrame con la información extraída
    df_info = pd.DataFrame({
        'file': v_file_name,
        'sex': v_sex,
        'age': v_age,
        'WAB_AQ': v_WAB_AQ,
        'aphasia_type': v_aphasia_type
    })

    # Aplicar reglas para WAB_AQ_category
    df_info.loc[(pd.to_numeric(df_info['WAB_AQ'], errors='coerce') >= 0) & (pd.to_numeric(df_info['WAB_AQ'], errors='coerce') <= 25), 'WAB_AQ_category'] = 'Very severe'
    df_info.loc[(pd.to_numeric(df_info['WAB_AQ'], errors='coerce') > 25) & (pd.to_numeric(df_info['WAB_AQ'], errors='coerce') <= 50), 'WAB_AQ_category'] = 'Severe'
    df_info.loc[(pd.to_numeric(df_info['WAB_AQ'], errors='coerce') > 50) & (pd.to_numeric(df_info['WAB_AQ'], errors='coerce') <= 75), 'WAB_AQ_category'] = 'Moderate'
    df_info.loc[(pd.to_numeric(df_info['WAB_AQ'], errors='coerce') > 75), 'WAB_AQ_category'] = 'Mild'
    df_info.loc[pd.to_numeric(df_info['WAB_AQ'], errors='coerce').isna(), 'WAB_AQ_category'] = 'Unknown'

    # Aplicar reglas para fluency_speech
    df_info.loc[df_info['aphasia_type'].isin(['Anomic', 'Conduction', 'Fluent', 'Wernicke', 'TransSensory']), 'fluency_speech'] = 'Fluent'
    df_info.loc[df_info['aphasia_type'].isin(['Broca', 'Global', 'TransMotor']), 'fluency_speech'] = 'Non Fluent'
    df_info.loc[df_info['aphasia_type'] == 'NotAphasicByWAB', 'fluency_speech'] = 'Unknown'

    return df_info

In [ ]:
path_data = '/content/drive/MyDrive/tesis_monica/afasia/data/'
file_directory = path_data + 'aphasiabank_es'
df_info = extract_patient_info(file_directory)
df_info

,file,sex,age,WAB_AQ,aphasia_type,WAB_AQ_category,fluency_speech
0,TCU02a.wav,male,42,77.3,Anomic,Mild,Fluent
1,TCU04a.wav,male,48,83.8,Anomic,Mild,Fluent
2,TCU06a.wav,male,68,62.6,,Moderate,NaN
3,TCU10a.wav,female,53,77.6,,Mild,NaN


In [ ]:
df_aphbank = pd.read_csv(path_data + 'df_ES_clean_2.csv')

# Combinar ambos datasets por la columna 'file'
df_combined = pd.merge(df_aphbank, df_info, on='file', how='left', suffixes=('', '_new'))

# Llenar los valores NaN del dataset original con los nuevos valores extraídos
df_combined['sex'] = df_combined['sex'].fillna(df_combined['sex_new'])
df_combined['age'] = df_combined['age'].fillna(df_combined['age_new'])
df_combined['WAB_AQ'] = df_combined['WAB_AQ'].fillna(df_combined['WAB_AQ_new'])
df_combined['aphasia_type'] = df_combined['aphasia_type'].fillna(df_combined['aphasia_type_new'])
df_combined['WAB_AQ_category'] = df_combined['WAB_AQ_category'].fillna(df_combined['WAB_AQ_category_new'])
df_combined['fluency_speech'] = df_combined['fluency_speech'].fillna(df_combined['fluency_speech_new'])

# Eliminar columnas adicionales generadas durante la combinación
df_combined = df_combined.drop(columns=[
    'sex_new', 'age_new', 'WAB_AQ_new', 'aphasia_type_new', 'WAB_AQ_category_new', 'fluency_speech_new'
])
df_combined

,mark_start,mark_end,transcriptions,sex,age,file,WAB_AQ,aphasia_type,WAB_AQ_category,fluency_speech,file_cut,duration,num_words
0,20585,24500,okay dímelo otra vez,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_20.585_3.915.wav,3.915,4
1,25710,26400,sí,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_25.71_0.69.wav,0.690,1
2,29400,31200,que piensa mi habla,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_29.4_1.8.wav,1.800,4
3,33248,40362,no es que flr flr decir las cosas a vveces ten...,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_33.248_7.114.wav,7.114,14
4,40362,47984,ahorita puedo hablar pero flr dos horas flr fl...,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_40.362_7.622.wav,7.622,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...
810,1922351,1932580,escoge el pan flr y el peanut butters úntelo e...,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1922.351_10.229.wav,10.229,12
811,1932580,1945033,y después en el otro pan úntelo en el jellys j...,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1932.58_12.453.wav,12.453,11
812,1945033,1949627,y juntelo y ya está,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1945.033_4.594.wav,4.594,5
813,1953138,1954175,lau,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1953.138_1.037.wav,1.037,1


In [ ]:
# Mapeos definidos para convertir texto en números
mapeo_aphasia_type = {
    "Anomic": 5,
    "Conduction": 4,
    "Fluent": 2,
    "Wernicke": 7,
    "TransSensory": 7,
    "Broca": 1,
    "Global": 3,
    "TransMotor": 6,
    "NotAphasicByWAB": 0
}

mapeo_fluency_speech = {
    "Fluent": "Fluente",
    "Non Fluent": "No Fluente",
    "Unknown": -1
}

# Mapeo definido para el género
mapeo_genere = {
    "male": 1,
    "female": 2
}

df_combined['sex_numeric'] = df_combined['sex'].map(mapeo_genere)
df_combined['TipusAfàsia'] = df_combined['aphasia_type'].map(mapeo_aphasia_type)
df_combined['fluency_speech_numeric'] = df_combined['fluency_speech'].map(mapeo_fluency_speech)
df_combined['LLengWAB'] = 2
df_combined['CIP'] = df_combined['file'].str.replace('.wav', '', regex=False)

df_combined

,mark_start,mark_end,transcriptions,sex,age,file,WAB_AQ,aphasia_type,WAB_AQ_category,fluency_speech,file_cut,duration,num_words,sex_numeric,TipusAfàsia,fluency_speech_numeric,LLengWAB,CIP
0,20585,24500,okay dímelo otra vez,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_20.585_3.915.wav,3.915,4,1,5.0,Fluente,2,TCU02a
1,25710,26400,sí,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_25.71_0.69.wav,0.690,1,1,5.0,Fluente,2,TCU02a
2,29400,31200,que piensa mi habla,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_29.4_1.8.wav,1.800,4,1,5.0,Fluente,2,TCU02a
3,33248,40362,no es que flr flr decir las cosas a vveces ten...,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_33.248_7.114.wav,7.114,14,1,5.0,Fluente,2,TCU02a
4,40362,47984,ahorita puedo hablar pero flr dos horas flr fl...,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_40.362_7.622.wav,7.622,14,1,5.0,Fluente,2,TCU02a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
810,1922351,1932580,escoge el pan flr y el peanut butters úntelo e...,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1922.351_10.229.wav,10.229,12,2,NaN,NaN,2,TCU10a
811,1932580,1945033,y después en el otro pan úntelo en el jellys j...,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1932.58_12.453.wav,12.453,11,2,NaN,NaN,2,TCU10a
812,1945033,1949627,y juntelo y ya está,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1945.033_4.594.wav,4.594,5,2,NaN,NaN,2,TCU10a
813,1953138,1954175,lau,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1953.138_1.037.wav,1.037,1,2,NaN,NaN,2,TCU10a


In [ ]:
column_mapping = {
    'mark_start': 'Inicio',
    'mark_end': 'Fin',
    'file': 'Transcrip_name',
    'transcriptions': 'Marca',
    'sex_numeric': 'Gènere',
    'age': 'Edat',
    'file_cut': 'name_chunk_audio',
    'WAB_AQ': 'QA',
    'WAB_AQ_category': 'Grup',
    'fluency_speech_numeric': 'Fluente/No Fluente',
    'duration': 'Duración'
}

# Renombrar las columnas del DataFrame df_aphbank
df_combined.rename(columns=column_mapping, inplace=True)

path_data = '/content/drive/MyDrive/tesis_monica/afasia/data/'
audio_base_path = path_data + 'aphasiabank_es/Audios_ES/'
# Crear la columna name_chunk_audio_path concatenando la ruta base con name_chunk_audio
df_combined['name_chunk_audio_path'] = audio_base_path + df_combined['name_chunk_audio']

df_combined['Transcrip_name'] = df_combined['Transcrip_name'].str.replace('.wav', '', regex=False)

df_combined[['Inicio', 'Fin', 'Marca', 'Transcrip_name', 'Duración',
       'name_chunk_audio', 'name_chunk_audio_path', 'Gènere',
       'TipusAfàsia', 'Edat', 'Grup', 'QA', 'Fluente/No Fluente']]


,Inicio,Fin,Marca,Transcrip_name,Duración,name_chunk_audio,name_chunk_audio_path,Gènere,TipusAfàsia,Edat,Grup,QA,Fluente/No Fluente
0,20585,24500,okay dímelo otra vez,TCU02a,3.915,TCU02a_20.585_3.915.wav,/content/drive/MyDrive/tesis_monica/afasia/dat...,1,5.0,42.0,Mild,77.3,Fluente
1,25710,26400,sí,TCU02a,0.690,TCU02a_25.71_0.69.wav,/content/drive/MyDrive/tesis_monica/afasia/dat...,1,5.0,42.0,Mild,77.3,Fluente
2,29400,31200,que piensa mi habla,TCU02a,1.800,TCU02a_29.4_1.8.wav,/content/drive/MyDrive/tesis_monica/afasia/dat...,1,5.0,42.0,Mild,77.3,Fluente
3,33248,40362,no es que flr flr decir las cosas a vveces ten...,TCU02a,7.114,TCU02a_33.248_7.114.wav,/content/drive/MyDrive/tesis_monica/afasia/dat...,1,5.0,42.0,Mild,77.3,Fluente
4,40362,47984,ahorita puedo hablar pero flr dos horas flr fl...,TCU02a,7.622,TCU02a_40.362_7.622.wav,/content/drive/MyDrive/tesis_monica/afasia/dat...,1,5.0,42.0,Mild,77.3,Fluente
...,...,...,...,...,...,...,...,...,...,...,...,...,...
810,1922351,1932580,escoge el pan flr y el peanut butters úntelo e...,TCU10a,10.229,TCU10a_1922.351_10.229.wav,/content/drive/MyDrive/tesis_monica/afasia/dat...,2,NaN,53,Unknown,77.6,NaN
811,1932580,1945033,y después en el otro pan úntelo en el jellys j...,TCU10a,12.453,TCU10a_1932.58_12.453.wav,/content/drive/MyDrive/tesis_monica/afasia/dat...,2,NaN,53,Unknown,77.6,NaN
812,1945033,1949627,y juntelo y ya está,TCU10a,4.594,TCU10a_1945.033_4.594.wav,/content/drive/MyDrive/tesis_monica/afasia/dat...,2,NaN,53,Unknown,77.6,NaN
813,1953138,1954175,lau,TCU10a,1.037,TCU10a_1953.138_1.037.wav,/content/drive/MyDrive/tesis_monica/afasia/dat...,2,NaN,53,Unknown,77.6,NaN


In [ ]:
path_data = '/content/drive/MyDrive/tesis_monica/afasia/data/'
df_combined.to_csv(path_data + 'df_ES_clean_updated.csv', index=False)

# aphasiabank eng

In [ ]:
path_data = '/content/drive/MyDrive/tesis_monica/afasia/data/'
file_path = path_data + 'df_paper_clean.csv'
df_aphbank_eng = pd.read_csv(file_path)

In [ ]:
df_aphbank_eng["file_cut"] = df_aphbank_eng["file"].str.replace(".wav", "", regex=True) + "_" + (df_aphbank_eng["mark_start"] / 1000).round(3).astype(str) + "_" + ((df_aphbank_eng["mark_end"] - df_aphbank_eng["mark_start"]) / 1000).round(3).astype(str) + ".wav"
df_aphbank_eng.loc[((df_aphbank_eng['aphasia_type'])== 'Anomic') | ((df_aphbank_eng['aphasia_type'])== 'Conduction') | ((df_aphbank_eng['aphasia_type'])== 'Fluent')| ((df_aphbank_eng['aphasia_type'])== 'Wernicke')| ((df_aphbank_eng['aphasia_type'])== 'TransSensory'), 'fluency_speech'] = 'Fluent'
df_aphbank_eng.loc[((df_aphbank_eng['aphasia_type'])== 'Broca') | ((df_aphbank_eng['aphasia_type'])== 'Global') | ((df_aphbank_eng['aphasia_type'])== 'TransMotor'), 'fluency_speech'] = 'Non Fluent'
df_aphbank_eng.loc[((df_aphbank_eng['aphasia_type'])== 'NotAphasicByWAB') , 'fluency_speech'] = 'Unknown'

df_aphbank_eng.loc[(pd.to_numeric(df_aphbank_eng['WAB_AQ'])>= 0) & (pd.to_numeric(df_aphbank_eng['WAB_AQ'])<=25), 'WAB_AQ_category'] = 'Very severe'
df_aphbank_eng.loc[(pd.to_numeric(df_aphbank_eng['WAB_AQ'])> 25) & (pd.to_numeric(df_aphbank_eng['WAB_AQ'])<=50), 'WAB_AQ_category'] = 'Severe'
df_aphbank_eng.loc[(pd.to_numeric(df_aphbank_eng['WAB_AQ'])> 50) & (pd.to_numeric(df_aphbank_eng['WAB_AQ'])<=75), 'WAB_AQ_category'] = 'Moderate'
df_aphbank_eng.loc[(pd.to_numeric(df_aphbank_eng['WAB_AQ'])> 75) , 'WAB_AQ_category'] = 'Mild'
df_aphbank_eng.loc[np.isnan(pd.to_numeric(df_aphbank_eng['WAB_AQ'])) , 'WAB_AQ_category'] = 'Unknown'

df_aphbank_eng['duration'] = df_aphbank_eng['mark_end'] - df_aphbank_eng['mark_start']

df_aphbank_eng['sex_numeric'] = df_aphbank_eng['sex'].map(mapeo_genere)
df_aphbank_eng['TipusAfàsia'] = df_aphbank_eng['aphasia_type'].map(mapeo_aphasia_type)
df_aphbank_eng['fluency_speech_numeric'] = df_aphbank_eng['fluency_speech'].map(mapeo_fluency_speech)
df_aphbank_eng['LLengWAB'] = 2
df_aphbank_eng['CIP'] = df_aphbank_eng['file'].str.replace('.wav', '', regex=False)

path_data = '/content/drive/MyDrive/tesis_monica/afasia/data/'
# audio_base_path = path_data + 'aphasiabank_es/chunks_3s/'
audio_base_path = path_data + '/aphasibank_en/'

df_aphbank_eng['name_chunk_audio_path'] = audio_base_path + df_aphbank_eng['file_cut']

df_aphbank_eng

,mark_start,mark_end,transcriptions,sex,age,file,WAB_AQ,aphasia_type,file_cut,fluency_speech,WAB_AQ_category,duration,sex_numeric,TipusAfàsia,fluency_speech_numeric,LLengWAB,CIP,name_chunk_audio_path
0,9430.0,12876.0,yeah well <lau>,female,69.0,ACWT01a.wav,63.9,Broca,ACWT01a_9.43_3.446.wav,Non Fluent,Moderate,3446.0,2.0,1.0,No Fluente,2,ACWT01a,/content/drive/MyDrive/tesis_monica/afasia/dat...
1,15181.0,20445.0,i yeah you know <flr> <flr> <flr> <lau>,female,69.0,ACWT01a.wav,63.9,Broca,ACWT01a_15.181_5.264.wav,Non Fluent,Moderate,5264.0,2.0,1.0,No Fluente,2,ACWT01a,/content/drive/MyDrive/tesis_monica/afasia/dat...
2,24756.0,33655.0,yes <flr> it's two thousand two days,female,69.0,ACWT01a.wav,63.9,Broca,ACWT01a_24.756_8.899.wav,Non Fluent,Moderate,8899.0,2.0,1.0,No Fluente,2,ACWT01a,/content/drive/MyDrive/tesis_monica/afasia/dat...
3,33655.0,34070.0,no,female,69.0,ACWT01a.wav,63.9,Broca,ACWT01a_33.655_0.415.wav,Non Fluent,Moderate,415.0,2.0,1.0,No Fluente,2,ACWT01a,/content/drive/MyDrive/tesis_monica/afasia/dat...
4,34070.0,43822.0,after <flr> new year's day two thousand,female,69.0,ACWT01a.wav,63.9,Broca,ACWT01a_34.07_9.752.wav,Non Fluent,Moderate,9752.0,2.0,1.0,No Fluente,2,ACWT01a,/content/drive/MyDrive/tesis_monica/afasia/dat...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76602,909216.0,911714.0,<flr> fold it,female,63.0,wright207a.wav,61.5,Broca,wright207a_909.216_2.498.wav,Non Fluent,Moderate,2498.0,2.0,1.0,No Fluente,2,wright207a,/content/drive/MyDrive/tesis_monica/afasia/dat...
76603,911714.0,914909.0,it's pretty good,female,63.0,wright207a.wav,61.5,Broca,wright207a_911.714_3.195.wav,Non Fluent,Moderate,3195.0,2.0,1.0,No Fluente,2,wright207a,/content/drive/MyDrive/tesis_monica/afasia/dat...
76604,914909.0,915458.0,yeah,female,63.0,wright207a.wav,61.5,Broca,wright207a_914.909_0.549.wav,Non Fluent,Moderate,549.0,2.0,1.0,No Fluente,2,wright207a,/content/drive/MyDrive/tesis_monica/afasia/dat...
76605,915458.0,917189.0,yeah pretty good,female,63.0,wright207a.wav,61.5,Broca,wright207a_915.458_1.731.wav,Non Fluent,Moderate,1731.0,2.0,1.0,No Fluente,2,wright207a,/content/drive/MyDrive/tesis_monica/afasia/dat...


In [ ]:
column_mapping = {
    'mark_start': 'Inicio',
    'mark_end': 'Fin',
    'file': 'Transcrip_name',
    'transcriptions': 'Marca',
    'sex_numeric': 'Gènere',
    'age': 'Edat',
    'file_cut': 'name_chunk_audio',
    'WAB_AQ': 'QA',
    'WAB_AQ_category': 'Grup',
    'fluency_speech_numeric': 'Fluente/No Fluente',
    'duration': 'Duración'
}

# Renombrar las columnas del DataFrame df_aphbank
df_aphbank_eng.rename(columns=column_mapping, inplace=True)
df_aphbank_eng.columns

Index(['Inicio', 'Fin', 'Marca', 'sex', 'Edat', 'Transcrip_name', 'QA',
       'aphasia_type', 'name_chunk_audio', 'fluency_speech', 'Grup',
       'Duración', 'Gènere', 'TipusAfàsia', 'Fluente/No Fluente', 'LLengWAB',
       'CIP', 'name_chunk_audio_path'],
      dtype='object')

In [ ]:
df_combined = df_combined.drop(columns=['num_words'], errors='ignore')
df_final = pd.concat([df_aphbank_eng, df_combined], ignore_index=True)

In [ ]:
path_data = '/content/drive/MyDrive/tesis_monica/afasia/data/'
df_final.to_csv(path_data + 'df_aphasiabank_all.csv', index=False)